# Physical Templates Prototype

**Purpose**: Validate the physical SED pipeline (MOSFiT magnetar chain via `mosfit_interface.py`)
before connecting it to production metric evaluation.

**Branch**: `tabulated-rate-model`

## What this notebook does
1. Builds physical SED templates from `all_parameters.txt` (265 Gómez+2024 events)
2. Builds a physical magnitude grid over `(template, z, phase, filter)`
3. Verifies UV coverage is present at z > 1.2 (the GP wall)
4. Validates low-z agreement between physical and GP templates

## What this notebook does NOT touch
- `templates.pkl` — existing GP templates (read-only for comparison only)
- `mag_grid.pkl` — existing GP mag grid (read-only for comparison only)
- Any production `.npy` or `.csv` result files
- `population_*.pkl` files

## Reload guide
| Component | Rebuild when | Toggle |
|---|---|---|
| `physical_templates.pkl` | `mosfit_interface.py` logic changes | `REBUILD_PHYSICAL_TEMPLATES` |
| `physical_mag_grid.pkl` | Physical templates change, or z/phase grid changes | `REBUILD_PHYSICAL_MAG_GRID` |
| `physical_sed_cache.pkl` | `gomez_models.py` physics changes | `REBUILD_SED_CACHE` |
| GP templates (`templates.pkl`) | **NEVER** from this notebook | — |
| GP mag grid (`mag_grid.pkl`) | **NEVER** from this notebook | — |
| Population | Not in scope here | — |
| Kernel | Not in scope here | — |

---
## Cell 0: Environment Check

Confirms all required imports are available before any computation.
If this cell fails, fix the import error before proceeding.

In [1]:
# ============================================================================
# CELL 0: Environment Check
# Run this first. Fix any ImportError before proceeding.
# ============================================================================
import sys
import importlib

REPO_ROOT = '/users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric'
PKG_PATH  = f'{REPO_ROOT}/py_files'

if PKG_PATH not in sys.path:
    sys.path.insert(0, PKG_PATH)

# ── Standard imports ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend safe for MSI
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
print(f'numpy  : {np.__version__}')
print(f'python : {sys.version}')

# ── rubin_sim ────────────────────────────────────────────────────────────────
try:
    from slsn_metrics.paths import get_repo_root, get_shared_output_dir, set_rubin_sim_data_dir
    set_rubin_sim_data_dir("/users/1/andra104/rubin_sim_data")
    print('paths  : OK')
except ImportError as e:
    raise ImportError(f'[FAIL] slsn_metrics.paths: {e}') from e

# ── model.py ────────────────────────────────────────────────────────────────
try:
    from slsn_metrics.model import LC, synthesize_mag_at_z
    print('model  : OK  (LC, synthesize_mag_at_z)')
except ImportError as e:
    raise ImportError(f'[FAIL] slsn_metrics.model: {e}') from e

# ── mosfit_interface.py ──────────────────────────────────────────────────────
# This file exists on the tabulated-rate-model branch in py_files/slsn_metrics/
# If this fails: confirm branch, confirm file exists at
#   SLSNe_Metric/py_files/slsn_metrics/mosfit_interface.py
try:
    from slsn_metrics.mosfit_interface import build_physical_templates
    print('mosfit : OK  (build_physical_templates)')
except ImportError as e:
    raise ImportError(
        f'[FAIL] slsn_metrics.mosfit_interface: {e}\n'
        f'Check: git branch should be tabulated-rate-model\n'
        f'Check: file exists at {PKG_PATH}/slsn_metrics/mosfit_interface.py'
    ) from e

# ── numexpr (required by gomez_models.py) ────────────────────────────────────
try:
    import numexpr
    print(f'numexpr: OK  ({numexpr.__version__})')
except ImportError:
    raise ImportError('[FAIL] numexpr not available. Run: conda install numexpr')

# ── extinction (required by mosfit_interface.py) ─────────────────────────────
try:
    import extinction
    print(f'extinction: OK')
except ImportError:
    raise ImportError('[FAIL] extinction not available. Run: pip install extinction')

print('\n✓ All imports OK — safe to proceed')

numpy  : 2.4.2
python : 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
[paths] RUBIN_SIM_DATA_DIR = /users/1/andra104/rubin_sim_data
paths  : OK
model  : OK  (LC, synthesize_mag_at_z)
mosfit : OK  (build_physical_templates)
numexpr: OK  (2.14.1)
extinction: OK

✓ All imports OK — safe to proceed


---
## Cell 1: Configuration

**All file paths and toggle flags live here.**
Do not hardcode paths in any later cell.

### Toggle flags
- `REBUILD_PHYSICAL_TEMPLATES = True` — re-runs `build_physical_templates()` even if `physical_templates.pkl` exists
- `REBUILD_PHYSICAL_MAG_GRID = True` — re-runs `build_magnitude_grid()` even if `physical_mag_grid.pkl` exists
- `REBUILD_SED_CACHE = True` — forces `slsnni()` to re-run for all 265 events; set False on reruns

**Normal rerun (no changes)**: all three False → loads from disk, skips computation.

**First run ever**: all three must be True.

In [ ]:
# ============================================================================
# CELL 1: Configuration
# ALL paths and toggles live here. Edit here only.
# ============================================================================

# ── Rebuild toggles ──────────────────────────────────────────────────────────
# Set True on first run. Set False on subsequent reruns to load from cache.
REBUILD_PHYSICAL_TEMPLATES  = False  # already built
REBUILD_PHYSICAL_MAG_GRID   = True   # needs building
REBUILD_SED_CACHE           = False  # already built

# ── Input files ──────────────────────────────────────────────────────────────
# all_parameters.txt: MOSFiT posterior medians for 265 Gómez+2024 events
# This file must have a 'texplosion_med' column (verified in last session smoke test)
ALL_PARAMS_FILE = Path('/users/1/andra104/Documents/Research/Transient_Metrics/'
                       'SLSNe_Metric/SLSNe/slsne/ref_data/all_parameters.txt')

# Existing GP templates and mag grid — READ ONLY
GP_TEMPLATES_FILE = Path('/users/1/andra104/Documents/Research/Transient_Metrics/'
                         'SLSNe_Metric/output/SLSNe/shared/templates.pkl')
GP_MAG_GRID_FILE  = Path('/users/1/andra104/Documents/Research/Transient_Metrics/'
                         'SLSNe_Metric/output/SLSNe/SLSNe_den_1e-07_d_None-None_Mpc_z_0.1-2.0_ext_True_kcor_False_None_mag_grid.pkl')

# ── Output files (new — safe to write) ───────────────────────────────────────
SHARED_DIR = Path('/users/1/andra104/Documents/Research/Transient_Metrics/'
                  'SLSNe_Metric/output/SLSNe/shared')
SHARED_DIR.mkdir(parents=True, exist_ok=True)

PHYSICAL_TEMPLATES_FILE = SHARED_DIR / 'physical_templates.pkl'
PHYSICAL_MAG_GRID_FILE  = SHARED_DIR / 'physical_mag_grid.pkl'
PHYSICAL_SED_CACHE_FILE = SHARED_DIR / 'physical_sed_cache.pkl'

# ── Magnitude grid parameters ────────────────────────────────────────────────
# Full science grid — validated against rate model (z to 5.0) and SED phase range
Z_GRID = np.linspace(0.02, 5.0, 100) #full science range

# phase_grid: physical templates can have pre-peak UV phases
# Use None here to let build_magnitude_grid() auto-detect from SED grids
# Not necessary to use None as there is no pre-peak data for the physical SEDs dont have prepeak data 
# (auto-detect reads the actual phase range from mosfit_interface output)
PHASE_GRID = np.concatenate([
    np.geomspace(1.0,  100.0, 50),   # dense near peak
    np.linspace(100.0, 400.0, 30),   # sparser at late times
])  # 80 points — estimated ~5-6 hours on MSI
FILTERS = list('ugrizy')

# ── Validation parameters ────────────────────────────────────────────────────
VALIDATION_Z_LOWZ   = 0.3   # low-z: physical vs GP should agree within ~0.5 mag
VALIDATION_Z_HIGHZ  = 1.5   # high-z: physical should be finite, GP should be NaN
VALIDATION_FILTER   = 'g'   # g-band for all comparisons
N_PLOT_EVENTS       = 3     # number of events to plot in Cell 7

# ── Confirm paths ────────────────────────────────────────────────────────────
print('=== Path Check ===')
print(f'all_parameters.txt : {ALL_PARAMS_FILE}  exists={ALL_PARAMS_FILE.exists()}')
print(f'GP templates       : {GP_TEMPLATES_FILE}  exists={GP_TEMPLATES_FILE.exists()}')
print(f'GP mag grid        : {GP_MAG_GRID_FILE}  exists={GP_MAG_GRID_FILE.exists()}')
print()
print(f'Physical templates : {PHYSICAL_TEMPLATES_FILE}  exists={PHYSICAL_TEMPLATES_FILE.exists()}')
print(f'Physical mag grid  : {PHYSICAL_MAG_GRID_FILE}  exists={PHYSICAL_MAG_GRID_FILE.exists()}')
print(f'SED cache          : {PHYSICAL_SED_CACHE_FILE}  exists={PHYSICAL_SED_CACHE_FILE.exists()}')
print()
print(f'Rebuild toggles    : templates={REBUILD_PHYSICAL_TEMPLATES}  '
      f'mag_grid={REBUILD_PHYSICAL_MAG_GRID}  sed_cache={REBUILD_SED_CACHE}')

# Guard: if ALL_PARAMS_FILE is missing, stop here — nothing else will work
if not ALL_PARAMS_FILE.exists():
    raise FileNotFoundError(
        f'all_parameters.txt not found at:\n  {ALL_PARAMS_FILE}\n'
        f'Check the path — this file contains the MOSFiT posterior medians.'
    )

---
## Cell 2: Build or Load Physical Templates

Calls `build_physical_templates()` from `mosfit_interface.py`.

### What `build_physical_templates()` does internally
1. Reads `all_parameters.txt` (265 events)
2. For each event: reads `texplosion_med` column (NOT hardcoded to 0.0 — earlier smoke test showed all-NaN when texplosion=0)
3. Calls `slsnni()` (Gómez+2024 magnetar physics, vendored in `gomez_models.py` with NumPy 2.0 fix)
4. Builds physical SED grid: F_λ from 500–12000 Å (covers rest-frame UV 500–3000 Å)
5. Converts to F_ν in Jy at 10 pc
6. Returns an `LC`-compatible object with `.sed_grid` populated

### Cache guard
- If `REBUILD_SED_CACHE = False` and `physical_sed_cache.pkl` exists, `slsnni()` is NOT re-run
- If `REBUILD_PHYSICAL_TEMPLATES = False` and `physical_templates.pkl` exists, this cell loads from disk

### Expected output
- `phys_model` — LC object with `.sed_grid` (list of 265 dicts, one per event)
- Each `sed_grid` entry: `{'phase': array, 'lam_rest_A': array, 'Fnu_abs': array[N_phase, N_lam]}`

### T=0 note
`_call_slsnni_safe()` in `mosfit_interface.py` already masks T=0 phases. Do not bypass this.

In [3]:
# ============================================================================
# CELL 2: Build or Load Physical Templates
# Output: phys_model (LC instance with .sed_grid populated)
# ============================================================================

if REBUILD_PHYSICAL_TEMPLATES or not PHYSICAL_TEMPLATES_FILE.exists():
    print('[PHYSICAL TEMPLATES] Building from all_parameters.txt ...')
    print(f'  Input : {ALL_PARAMS_FILE}')
    print(f'  Cache : {PHYSICAL_SED_CACHE_FILE}  (rebuild_cache={REBUILD_SED_CACHE})')
    print(f'  Output: {PHYSICAL_TEMPLATES_FILE}')
    print()

    # build_physical_templates() actual signature (mosfit_interface.py):
    #   build_physical_templates(params_file, save_to, phase_grid,
    #                            wave_grid_A, verbose, cache_file) -> LC
    # No rebuild_cache param — delete the cache file to force a rebuild.

    if REBUILD_SED_CACHE and PHYSICAL_SED_CACHE_FILE.exists():
        PHYSICAL_SED_CACHE_FILE.unlink()
        print(f'  [cache] Deleted cache file to force rebuild: {PHYSICAL_SED_CACHE_FILE}')

    phys_model = build_physical_templates(
        params_file = ALL_PARAMS_FILE,
        save_to     = PHYSICAL_TEMPLATES_FILE,
        cache_file  = PHYSICAL_SED_CACHE_FILE,
    )
    print(f'\n[PHYSICAL TEMPLATES] Built: {len(phys_model.names)} events')

else:
    print(f'[PHYSICAL TEMPLATES] Loading from {PHYSICAL_TEMPLATES_FILE} ...')

    # Physical templates are saved via joblib (build_physical_templates uses joblib.dump)
    # The payload structure mirrors from_catalog output:
    #   {'lightcurves': ..., 't_grid': ..., 'names': ..., 'sed_grid': ..., 'meta': ...}
    import joblib
    payload = joblib.load(PHYSICAL_TEMPLATES_FILE)

    phys_model = LC(
        lightcurves = payload['lightcurves'],
        t_grid      = payload.get('t_grid'),
        names       = payload.get('names'),
    )
    phys_model.sed_grid = payload['sed_grid']
    print(f'[PHYSICAL TEMPLATES] Loaded: {len(phys_model.names)} events')

print(f'\n✓ phys_model ready')
print(f'  Events     : {len(phys_model.names)}')
print(f'  SED grids  : {len(phys_model.sed_grid) if phys_model.sed_grid else "MISSING"}')

[PHYSICAL TEMPLATES] Building from all_parameters.txt ...
  Input : /users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/all_parameters.txt
  Cache : /users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric/output/SLSNe/shared/physical_sed_cache.pkl  (rebuild_cache=True)
  Output: /users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric/output/SLSNe/shared/physical_templates.pkl



Building physical SED grids:   0%|          | 0/265 [00:00<?, ?it/s]

Building physical SED grids: 100%|██████████| 265/265 [00:17<00:00, 14.96it/s]



[PHYSICAL TEMPLATES] Built: 265 events

✓ phys_model ready
  Events     : 265
  SED grids  : 265


---
## Cell 3: Smoke Check — Physical Template SED Properties

Before building the mag grid, verify the SED grids look physically reasonable.

### What to check
1. **Count**: How many events have non-empty SED grids? (expect ~200–265 out of 265)
2. **Phase range**: Should span negative pre-peak phases to >100 days post-peak
3. **Wavelength range**: Must include rest-frame UV ≥ 500 Å (confirms `gomez_models.py` is running)
4. **`texplosion` check**: Print the `texplosion` range — should NOT be all 0.0
5. **UV flux check**: Confirm event 1991D has non-NaN F_ν at 1600–3000 Å at peak phase

If UV flux is all-NaN at this stage, the SED build failed — do not proceed to Cell 4.

In [6]:
# ============================================================================
# CELL 3: Smoke Check — Physical Template SED Properties
# Prerequisite: phys_model from Cell 2
# No reloads needed.
# ============================================================================

assert phys_model.sed_grid is not None, 'sed_grid is None — Cell 2 did not populate it'

sed_grids = phys_model.sed_grid
n_total   = len(sed_grids)

# ── 1. Count non-empty SED grids ─────────────────────────────────────────────
n_nonempty = sum(
    1 for s in sed_grids
    if s is not None
    and 'Fnu_abs' in s
    and np.any(np.isfinite(s['Fnu_abs']))
)
print(f'[CHECK 1] Non-empty SED grids: {n_nonempty} / {n_total}')
if n_nonempty < 150:
    print(f'  WARNING: fewer than 150 valid SEDs — check mosfit_interface.py for errors')

# ── 2. Phase range across all templates ──────────────────────────────────────
all_phase_min, all_phase_max = [], []
for s in sed_grids:
    if s is not None and 'phase' in s:
        ph = np.asarray(s['phase'], float)
        all_phase_min.append(float(np.nanmin(ph)))
        all_phase_max.append(float(np.nanmax(ph)))

if all_phase_min:
    print(f'[CHECK 2] Phase range: [{min(all_phase_min):.1f}, {max(all_phase_max):.1f}] days')
    if min(all_phase_min) == 0.0:
        print('  WARNING: phase minimum is 0.0 — texplosion may be hardcoded to 0')
        print('  Check: all_parameters.txt must have texplosion_med column')
else:
    print('[CHECK 2] FAIL: no phase data found in any SED grid')

# ── 3. Wavelength range ───────────────────────────────────────────────────────
s0 = next(s for s in sed_grids if s is not None and 'lam_rest_A' in s)
lam = np.asarray(s0['lam_rest_A'], float)
print(f'[CHECK 3] Wavelength range: [{lam.min():.0f}, {lam.max():.0f}] Å')
if lam.min() > 1000:
    print('  WARNING: wavelength floor > 1000 Å — rest-frame UV below 1000 Å not covered')
    print('  This is OK if min is ~500 Å (covers LSST u-band rest-frame at z~1.2)')
if lam.min() > 3000:
    print('  ERROR: wavelength floor > 3000 Å — UV physics chain may have failed')

# ── 4. UV flux check (event 1991D, peak phase, 1600–3000 Å) ─────────────────
target_event = '1991D'   # name as stored in all_parameters.txt (no 'SN' prefix)
if target_event in phys_model.names:
    idx = phys_model.names.index(target_event)
    s   = sed_grids[idx]
    ph  = np.asarray(s['phase'], float)
    lam_arr = np.asarray(s['lam_rest_A'], float)
    Fnu = np.asarray(s['Fnu_abs'], float)

    # Find peak phase (max total flux)
    peak_idx = np.nanargmax(np.nansum(Fnu, axis=1))
    peak_ph  = ph[peak_idx]

    # UV slice: 1600–3000 Å
    uv_mask = (lam_arr >= 1600) & (lam_arr <= 3000)
    Fnu_uv  = Fnu[peak_idx, uv_mask]

    n_finite_uv = np.sum(np.isfinite(Fnu_uv))
    print(f'[CHECK 4] {target_event}: peak at phase={peak_ph:.1f} days')
    print(f'          UV (1600-3000A): {n_finite_uv}/{uv_mask.sum()} finite flux points')
    if n_finite_uv == 0:
        print('  ERROR: all UV flux is NaN — SED build failed for this event')
        print('  This is a STOP condition — do not proceed to Cell 4')
    else:
        uv_median = float(np.nanmedian(Fnu_uv))
        print(f'          UV median F_nu = {uv_median:.3e} Jy  (should be > 0)')
else:
    # 1991D not always present — try first event with valid SED
    print(f'[CHECK 4] {target_event} not found — checking first valid event instead')
    for i, (name, s) in enumerate(zip(phys_model.names, sed_grids)):
        if s is not None and 'Fnu_abs' in s and np.any(np.isfinite(s['Fnu_abs'])):
            lam_arr = np.asarray(s['lam_rest_A'], float)
            Fnu = np.asarray(s['Fnu_abs'], float)
            uv_mask = (lam_arr >= 1600) & (lam_arr <= 3000)
            Fnu_uv  = Fnu[0, uv_mask]  # first phase step
            n_finite_uv = np.sum(np.isfinite(Fnu_uv))
            print(f'  Event {name} (idx {i}): UV finite points = {n_finite_uv}/{uv_mask.sum()}')
            break

# ── 5. texplosion check (read from all_parameters.txt directly) ──────────────
import pandas as pd
try:
    params_df = pd.read_csv(ALL_PARAMS_FILE, sep=r'\s+', engine='python')
    # Normalize column names
    params_df.columns = [c.strip() for c in params_df.columns]
    texp_col = next((c for c in params_df.columns if 'texplosion' in c.lower()), None)
    if texp_col:
        t_vals = pd.to_numeric(params_df[texp_col], errors='coerce').dropna()
        print(f'[CHECK 5] texplosion column: "{texp_col}"')
        print(f'          range: [{t_vals.min():.1f}, {t_vals.max():.1f}]')
        n_zero = (t_vals == 0.0).sum()
        print(f'          events with texplosion=0.0: {n_zero}/{len(t_vals)}')
        if n_zero == len(t_vals):
            print('  ERROR: ALL texplosion values are 0.0 — will cause all-NaN SED output')
            print('  Check: mosfit_interface.py must read texplosion_med, not hardcode 0.0')
    else:
        print(f'[CHECK 5] WARNING: no texplosion column found in all_parameters.txt')
        print(f'          Columns: {list(params_df.columns[:10])} ...')
except Exception as e:
    print(f'[CHECK 5] Could not read all_parameters.txt: {e}')

print('\n=== Smoke check complete. Address any ERROR or WARNING above before Cell 4. ===')

[CHECK 1] Non-empty SED grids: 265 / 265
[CHECK 2] Phase range: [1.0, 400.0] days
[CHECK 3] Wavelength range: [500, 12000] Å
[CHECK 4] 1991D: peak at phase=10.0 days
          UV (1600-3000A): 365/365 finite flux points
          UV median F_nu = 6.473e-01 Jy  (should be > 0)
[CHECK 5] texplosion column: "texplosion_lo"
          range: [0.1, 81.7]
          events with texplosion=0.0: 0/265

=== Smoke check complete. Address any ERROR or WARNING above before Cell 4. ===


---
## Cell 4: Build or Load Physical Magnitude Grid

Calls `phys_model.build_magnitude_grid()` — the same method as GP templates use.
This is the slow step (~20–60 min on MSI for 265 events at 60 redshift points).

### Checkpointing
`build_magnitude_grid()` checkpoints every 50 templates to a `.tmp` file.
If the job dies, re-run Cell 4 — it will resume from the last checkpoint.

### Expected output shape
- `phys_model.mag_grid['g']` shape: `(N_events, 60, N_phase)`
- `phys_model.mag_grid_axes = {'z': Z_GRID, 'phase': <auto-detected>}`

### Do NOT rebuild when
- Switching between cadences (cadence is a property of the OpSim DB, not the mag grid)
- Switching between rate models (rate model affects population, not the mag grid)
- Running different metrics on the same population

### DO rebuild when
- Physical templates change (i.e., Cell 2 was re-run with different SED physics)
- Z_GRID or PHASE_GRID parameters change

In [ ]:
# ============================================================================
# CELL 4: Build or Load Physical Magnitude Grid
# Prerequisite: phys_model from Cell 2 with sed_grid populated
# Output: phys_model.mag_grid populated (in-place)
# ============================================================================

# Sanity check: sed_grid must be populated before building mag grid
if phys_model.sed_grid is None or len(phys_model.sed_grid) == 0:
    raise RuntimeError(
        'phys_model.sed_grid is empty — cannot build magnitude grid.\n'
        'Re-run Cell 2 with REBUILD_PHYSICAL_TEMPLATES=True.'
    )

if REBUILD_PHYSICAL_MAG_GRID or not PHYSICAL_MAG_GRID_FILE.exists():
    print('[PHYSICAL MAG GRID] Building ...')
    print(f'  Templates : {len(phys_model.names)} events')
    print(f'  z grid    : [{Z_GRID.min():.2f}, {Z_GRID.max():.2f}]  n={len(Z_GRID)}')
    print(f'  phase grid: auto-detect from SED grids' if PHASE_GRID is None else f'  phase grid: {PHASE_GRID}')
    print(f'  filters   : {FILTERS}')
    print(f'  Output    : {PHYSICAL_MAG_GRID_FILE}')
    print()
    print('  NOTE: This is the slow step. Checkpoint every 50 templates.')
    print('  If interrupted, re-run this cell — it resumes from checkpoint.')
    print()

    # build_magnitude_grid() is defined in model.py (LC class method).
    # With PHASE_GRID=None, it auto-detects phase range from sed_grid entries.
    # With save_to set, it does an atomic pickle write on completion.
    # With checkpoint_every=50, it saves intermediate state every 50 templates.
    phys_model.build_magnitude_grid(
        z_grid           = Z_GRID,
        phase_grid       = PHASE_GRID,    # None → auto-detect
        filters          = FILTERS,
        save_to          = PHYSICAL_MAG_GRID_FILE,
        checkpoint_every = 50,
    )
    print(f'\n[PHYSICAL MAG GRID] Built and saved → {PHYSICAL_MAG_GRID_FILE}')

else:
    print(f'[PHYSICAL MAG GRID] Loading from {PHYSICAL_MAG_GRID_FILE} ...')
    # load_magnitude_grid is defined in model.py — uses pickle.load
    # and calls _build_interpolators() automatically
    phys_model.load_magnitude_grid(PHYSICAL_MAG_GRID_FILE)
    print(f'[PHYSICAL MAG GRID] Loaded.')

# Confirm
grid_shape = phys_model.mag_grid[VALIDATION_FILTER].shape
print(f'\n✓ Physical mag grid ready')
print(f'  Shape ({VALIDATION_FILTER}-band): {grid_shape}')
print(f'  z range   : [{phys_model.mag_grid_axes["z"].min():.2f}, {phys_model.mag_grid_axes["z"].max():.2f}]')
print(f'  phase range: [{phys_model.mag_grid_axes["phase"].min():.1f}, {phys_model.mag_grid_axes["phase"].max():.1f}]')

---
## Cell 5: Validation A — High-z Coverage

**Test**: At z=1.5, g-band physical mag should be finite. GP mag should be NaN.

This is the core motivation for physical templates. The GP templates use empirical
photometry in B/V/R bands. At z=1.5, LSST g-band samples rest-frame ~2000 Å —
below any ground-based photometry in Gómez+2024. So GP → NaN is expected and correct.

Physical templates use the magnetar blackbody SED, which is defined at 500 Å and above.
So physical → finite is the success criterion.

### Pass/fail criteria
- **PASS**: ≥ 50% of physical templates give finite g-band mag at z=1.5, peak phase
- **FAIL**: < 10% finite — indicates SED wavelength coverage issue

In [ ]:
# ============================================================================
# CELL 5: Validation A — High-z Coverage (z=1.5)
# Prerequisite: phys_model.mag_grid from Cell 4
# ============================================================================

import warnings

# ── Sample physical mag grid at (z=1.5, peak phase, g-band) ──────────────────
# The mag grid is stored as absolute mags: m_abs[tpl, i_z, i_ph]
# We look up using the interpolator _interps built by _build_interpolators()

z_test     = VALIDATION_Z_HIGHZ   # 1.5
filt_test  = VALIDATION_FILTER    # 'g'

z_axis  = phys_model.mag_grid_axes['z']
ph_axis = phys_model.mag_grid_axes['phase']

# Find the index closest to z_test in the z grid
i_z = np.argmin(np.abs(z_axis - z_test))
z_actual = z_axis[i_z]
print(f'Querying at z={z_actual:.3f} (requested z={z_test})')

# Find peak phase index: use the phase closest to 0 (peak) for each template
# Peak is approximately at phase=0 in the rest-frame convention
i_ph_peak = np.argmin(np.abs(ph_axis - 0.0))
ph_peak   = ph_axis[i_ph_peak]
print(f'Peak phase index: i={i_ph_peak}  phase={ph_peak:.1f} days')

# Extract the g-band slice at (z_test, peak_phase) for all templates
# mag_grid[filt] shape: (N_tpl, N_z, N_phase)
phys_mags_highz = phys_model.mag_grid[filt_test][:, i_z, i_ph_peak]
n_finite_phys   = np.sum(np.isfinite(phys_mags_highz))
n_total_tpl     = len(phys_mags_highz)

print(f'\n[VALIDATION A] Physical templates at z={z_actual:.2f}, {filt_test}-band, peak phase')
print(f'  Finite mags : {n_finite_phys} / {n_total_tpl}  ({100*n_finite_phys/n_total_tpl:.0f}%)')
if n_finite_phys > 0:
    print(f'  Mag range   : [{np.nanmin(phys_mags_highz):.2f}, {np.nanmax(phys_mags_highz):.2f}]')
    print(f'  Median mag  : {np.nanmedian(phys_mags_highz):.2f}')

# Pass/fail
pct_finite = n_finite_phys / n_total_tpl
if pct_finite >= 0.5:
    print(f'  RESULT: PASS ✓ ({100*pct_finite:.0f}% ≥ 50%)')
elif pct_finite >= 0.1:
    print(f'  RESULT: MARGINAL — only {100*pct_finite:.0f}% finite (target ≥ 50%)')
    print('  Possible cause: some events lack UV flux in the SED grid')
else:
    print(f'  RESULT: FAIL ✗ ({100*pct_finite:.0f}% finite)')
    print('  This means the SED wavelength grid does not reach rest-frame UV at z=1.5')
    print('  Expected rest-frame λ for LSST g-band at z=1.5: ~1900 Å')
    print('  Check: lam_rest_A in sed_grid must extend to ≤ 1900 Å')

# ── Compare: GP templates at z=1.5 ──────────────────────────────────────────
# Load GP mag grid read-only for comparison
# GP grid is saved with atomic_save_pickle (plain pickle)
print()
if GP_MAG_GRID_FILE.exists():
    print(f'Loading GP mag grid for comparison (read-only)...')
    with open(GP_MAG_GRID_FILE, 'rb') as f:
        gp_grid_data = pickle.load(f)

    gp_mag_grid = gp_grid_data['mag_grid']
    gp_z_axis   = gp_grid_data['mag_grid_axes']['z']

    if filt_test in gp_mag_grid:
        # Find closest z in GP grid
        i_z_gp     = np.argmin(np.abs(gp_z_axis - z_test))
        z_gp_actual = gp_z_axis[i_z_gp]

        # GP mag grid phase axis
        gp_ph_axis = gp_grid_data['mag_grid_axes']['phase']
        i_ph_gp    = np.argmin(np.abs(gp_ph_axis - 0.0))

        gp_mags_highz = gp_mag_grid[filt_test][:, i_z_gp, i_ph_gp]
        n_finite_gp   = np.sum(np.isfinite(gp_mags_highz))

        print(f'[VALIDATION A] GP templates at z={z_gp_actual:.2f}, {filt_test}-band, peak phase')
        print(f'  Finite mags : {n_finite_gp} / {len(gp_mags_highz)}')
        if n_finite_gp == 0:
            print(f'  RESULT: GP returns all NaN at z={z_test} as expected ✓')
        else:
            print(f'  NOTE: GP has {n_finite_gp} finite mags at z={z_test} — unexpected')
    else:
        print(f'  {filt_test}-band not in GP mag grid keys: {list(gp_mag_grid.keys())}')
else:
    print(f'GP mag grid not found at {GP_MAG_GRID_FILE} — skipping GP comparison')

---
## Cell 6: Validation B — Low-z Agreement (z=0.3)

**Test**: At z=0.3, physical mag grid and GP mag grid should agree within ~0.5 mag
for most events at peak phase in g-band.

### Why 0.5 mag tolerance?
- Physical templates use posterior-median MOSFiT parameters → median light curve shape
- GP templates are direct fits to observed photometry → include scatter, upper limits, sparse sampling
- The two represent the same underlying events but through different modeling approaches
- Perfect agreement is NOT expected — 0.5 mag RMS is physically reasonable

### What counts as a "match"
- Same event name must exist in both template sets
- Both must give finite mag at z=0.3, peak phase, g-band
- |Δmag| < 0.5 is a match

In [ ]:
# ============================================================================
# CELL 6: Validation B — Low-z Agreement (z=0.3)
# Prerequisite: phys_model.mag_grid from Cell 4
#               gp_grid_data loaded in Cell 5 (or loaded here if Cell 5 skipped)
# ============================================================================

z_test    = VALIDATION_Z_LOWZ    # 0.3
filt_test = VALIDATION_FILTER    # 'g'

# Load GP templates for name matching (need names to cross-match)
# The GP templates pkl was saved with joblib (_atomic_joblib_dump) in from_catalog,
# but the names are accessible via pickle.load for payloads that use atomic_save_pickle.
# Try joblib first, fall back to pickle.
print('Loading GP templates (read-only, for name matching)...')
if not GP_TEMPLATES_FILE.exists():
    print(f'  GP templates file not found: {GP_TEMPLATES_FILE}')
    print('  Skipping low-z comparison.')
    gp_names = []
else:
    try:
        import joblib
        gp_payload = joblib.load(GP_TEMPLATES_FILE)
        gp_names = gp_payload.get('names', [])
        print(f'  Loaded via joblib: {len(gp_names)} GP templates')
    except Exception:
        with open(GP_TEMPLATES_FILE, 'rb') as f:
            gp_payload = pickle.load(f)
        gp_names = gp_payload.get('names', [])
        print(f'  Loaded via pickle: {len(gp_names)} GP templates')

# Load GP mag grid if not already in memory from Cell 5
if 'gp_grid_data' not in dir() and GP_MAG_GRID_FILE.exists():
    with open(GP_MAG_GRID_FILE, 'rb') as f:
        gp_grid_data = pickle.load(f)
elif not GP_MAG_GRID_FILE.exists():
    print(f'GP mag grid not found — skipping low-z comparison')
    gp_grid_data = None

if gp_grid_data is not None and len(gp_names) > 0:
    gp_z_axis  = gp_grid_data['mag_grid_axes']['z']
    gp_ph_axis = gp_grid_data['mag_grid_axes']['phase']
    gp_mag_g   = gp_grid_data['mag_grid'].get(filt_test)

    if gp_mag_g is None:
        print(f'{filt_test}-band not in GP mag grid — cannot compare')
    else:
        # Physical grid indices for z=0.3, peak phase
        phys_z_axis  = phys_model.mag_grid_axes['z']
        phys_ph_axis = phys_model.mag_grid_axes['phase']
        i_z_phys  = np.argmin(np.abs(phys_z_axis - z_test))
        i_ph_phys = np.argmin(np.abs(phys_ph_axis - 0.0))

        # GP grid indices for z=0.3, peak phase
        i_z_gp  = np.argmin(np.abs(gp_z_axis - z_test))
        i_ph_gp = np.argmin(np.abs(gp_ph_axis - 0.0))

        print(f'Physical: z={phys_z_axis[i_z_phys]:.3f}, phase={phys_ph_axis[i_ph_phys]:.1f}')
        print(f'GP      : z={gp_z_axis[i_z_gp]:.3f}, phase={gp_ph_axis[i_ph_gp]:.1f}')

        # Cross-match by event name
        delta_mags = []
        phys_names_set = set(phys_model.names)
        matched = 0
        both_finite = 0

        for gp_idx, name in enumerate(gp_names):
            if name not in phys_names_set:
                continue
            matched += 1
            phys_idx = phys_model.names.index(name)

            m_phys = phys_model.mag_grid[filt_test][phys_idx, i_z_phys, i_ph_phys]
            m_gp   = gp_mag_g[gp_idx, i_z_gp, i_ph_gp]

            if np.isfinite(m_phys) and np.isfinite(m_gp):
                both_finite += 1
                delta_mags.append(m_phys - m_gp)

        delta_mags = np.array(delta_mags)
        print(f'\n[VALIDATION B] Low-z agreement at z={z_test}')
        print(f'  Events in common    : {matched}')
        print(f'  Both finite in g    : {both_finite}')

        if len(delta_mags) > 0:
            rms   = float(np.sqrt(np.mean(delta_mags**2)))
            median = float(np.median(delta_mags))
            within_05 = np.sum(np.abs(delta_mags) < 0.5)

            print(f'  Δmag (phys - GP)    : median={median:+.2f}  RMS={rms:.2f}')
            print(f'  |Δmag| < 0.5 mag    : {within_05}/{len(delta_mags)}  ({100*within_05/len(delta_mags):.0f}%)')

            if rms < 0.5 and within_05/len(delta_mags) > 0.5:
                print(f'  RESULT: PASS ✓ (RMS < 0.5 and >50% within 0.5 mag)')
            elif rms < 1.0:
                print(f'  RESULT: MARGINAL — RMS={rms:.2f} mag (expected < 0.5)')
                print('  Acceptable if median offset is < 0.3 mag (systematic template offset)')
            else:
                print(f'  RESULT: FAIL ✗ — RMS={rms:.2f} mag suggests a systematic problem')
                print('  Possible causes:')
                print('    - texplosion mismatch (peak phase offset between GP and physical)')
                print('    - Normalization error in synthesize_mag_at_z()')

            # Histogram
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.hist(delta_mags, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
            ax.axvline(0, color='k', ls='-', lw=1.5, label='zero offset')
            ax.axvline(median, color='tomato', ls='--', lw=1.5, label=f'median={median:+.2f}')
            ax.axvline(-0.5, color='gray', ls=':', lw=1)
            ax.axvline(+0.5, color='gray', ls=':', lw=1, label='±0.5 mag')
            ax.set_xlabel(r'$\Delta m$ (physical $-$ GP)  [{} band, z={:.1f}]'.format(filt_test, z_test))
            ax.set_ylabel('Count')
            ax.set_title('Validation B: Low-z Mag Agreement')
            ax.legend(fontsize=9)
            plt.tight_layout()
            plt.savefig(SHARED_DIR / 'validation_B_lowz_agreement.png', dpi=120)
            plt.show()
            print(f'  Plot saved → {SHARED_DIR}/validation_B_lowz_agreement.png')
        else:
            print('  No matched events with both finite mags — cannot compare')
else:
    print('Skipping low-z comparison (GP data not loaded).')

---
## Cell 7: Validation C — Light Curve Comparison Plots

Plots 3 events showing physical vs GP light curves at z=0.3 in g-band.

### What to look for
- Physical curves should peak near phase=0 (some offset expected due to texplosion)
- Physical curves should show gradual decline after peak (not sharp cutoffs)
- GP curves should be present only where GP templates were fit (limited UV)
- Both should be in reasonable apparent magnitude range (~18–25 at z=0.3)

### Events selected
The 3 events are chosen by: (1) present in both template sets, (2) both give
finite mags in g at z=0.3, (3) selected by name alphabetically for reproducibility.

In [ ]:
# ============================================================================
# CELL 7: Validation C — Light Curve Comparison Plots
# Prerequisite: phys_model.mag_grid from Cell 4, gp_grid_data from Cell 5/6
#               gp_names from Cell 6
# ============================================================================

from slsn_metrics.constants import dm_from_z

z_plot    = VALIDATION_Z_LOWZ   # 0.3
filt_plot = VALIDATION_FILTER   # 'g'
DM_plot   = float(dm_from_z(np.array([z_plot]))[0])

print(f'Plotting light curves at z={z_plot}, DM={DM_plot:.2f} mag, {filt_plot}-band')

phys_z_axis  = phys_model.mag_grid_axes['z']
phys_ph_axis = phys_model.mag_grid_axes['phase']
i_z_phys     = np.argmin(np.abs(phys_z_axis - z_plot))

if 'gp_grid_data' not in dir() or gp_grid_data is None or len(gp_names) == 0:
    print('GP data not available — plotting physical templates only')
    plot_gp = False
else:
    plot_gp    = True
    gp_z_axis  = gp_grid_data['mag_grid_axes']['z']
    gp_ph_axis = gp_grid_data['mag_grid_axes']['phase']
    gp_mag_g   = gp_grid_data['mag_grid'].get(filt_plot)
    i_z_gp     = np.argmin(np.abs(gp_z_axis - z_plot))

# Select events to plot
# Strategy: find events in both sets with finite physical mag at z=0.3 peak phase
i_ph_phys_peak = np.argmin(np.abs(phys_ph_axis - 0.0))
phys_names_set = set(phys_model.names)

candidates = []
common_names = sorted([
    name for name in (gp_names if plot_gp else phys_model.names)
    if name in phys_names_set
])

for name in common_names:
    phys_idx = phys_model.names.index(name)
    m_phys_peak = phys_model.mag_grid[filt_plot][phys_idx, i_z_phys, i_ph_phys_peak]
    if np.isfinite(m_phys_peak):
        candidates.append(name)
    if len(candidates) >= N_PLOT_EVENTS:
        break

print(f'Selected events for plotting: {candidates}')

fig, axes = plt.subplots(1, len(candidates), figsize=(5*len(candidates), 5), sharey=True)
if len(candidates) == 1:
    axes = [axes]

for ax, name in zip(axes, candidates):
    phys_idx = phys_model.names.index(name)

    # Physical light curve: absolute mag + DM  (apparent mag at z_plot)
    phys_lc = phys_model.mag_grid[filt_plot][phys_idx, i_z_phys, :]  # shape (N_phase,)
    # phys_lc is absolute mag (stored without DM); add DM to get apparent
    phys_app = phys_lc + DM_plot
    valid_phys = np.isfinite(phys_app)

    ax.plot(
        phys_ph_axis[valid_phys], phys_app[valid_phys],
        color='steelblue', lw=2, label='Physical (MOSFiT)'
    )

    # GP light curve (if available)
    if plot_gp and gp_mag_g is not None and name in gp_names:
        gp_idx = gp_names.index(name)
        gp_lc  = gp_mag_g[gp_idx, i_z_gp, :]  # absolute mag
        gp_app = gp_lc + float(dm_from_z(np.array([gp_z_axis[i_z_gp]]))[0])
        valid_gp = np.isfinite(gp_app)
        ax.plot(
            gp_ph_axis[valid_gp], gp_app[valid_gp],
            color='tomato', lw=1.5, ls='--', label='GP (Gómez+2024)'
        )

    ax.axhline(24.7, color='gray', ls=':', lw=1, label='r~24.7 (approx WFD limit)')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Rest-frame phase (days)')
    ax.invert_yaxis()
    ax.legend(fontsize=8)

axes[0].set_ylabel(f'Apparent {filt_plot}-band mag  (z={z_plot})')
fig.suptitle(f'Validation C: Physical vs GP Light Curves  [z={z_plot}]', y=1.01)
plt.tight_layout()
plt.savefig(SHARED_DIR / 'validation_C_lc_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Plot saved → {SHARED_DIR}/validation_C_lc_comparison.png')

---
## Cell 8: Summary and Next Steps

Prints a final checklist of what passed and what needs attention.

In [ ]:
# ============================================================================
# CELL 8: Summary
# ============================================================================

print('=' * 60)
print('PROTOTYPE PHYSICAL TEMPLATES — RUN SUMMARY')
print('=' * 60)

print(f'\n[Files written]')
for label, path in [
    ('Physical templates', PHYSICAL_TEMPLATES_FILE),
    ('Physical mag grid ', PHYSICAL_MAG_GRID_FILE),
    ('SED cache         ', PHYSICAL_SED_CACHE_FILE),
    ('Val B plot        ', SHARED_DIR / 'validation_B_lowz_agreement.png'),
    ('Val C plot        ', SHARED_DIR / 'validation_C_lc_comparison.png'),
]:
    exists = Path(path).exists()
    print(f'  {label}: {"✓" if exists else "MISSING"}  {path}')

print()
print('[Files NOT touched (GP originals)]')
for label, path in [
    ('GP templates  ', GP_TEMPLATES_FILE),
    ('GP mag grid   ', GP_MAG_GRID_FILE),
]:
    print(f'  {label}: {path}')

print()
print('[Next steps — after validation passes]')
print('  1. Wire physical templates into metrics.py evaluate_slsn()')
print('     - evaluate_slsn() currently uses direct catalog-band interpolation')
print('     - Gap #3 in PIPELINE_GUIDE.md: synthesize_mag_at_z() not called in production')
print('     - This is a separate session — do not do it here')
print()
print('  2. Contact Sebastian Gómez about full MOSFiT fit outputs (all 168 gold SLSNe)')
print('     - Zenodo or his machine — needed to close K-correction question fully')
print()
print('  3. Tier 2 pipeline audit: caching/checkpoint strategy for mosfit_interface.py')
print('     - Currently no cache guard for 265-event slsnni() calls')
print('     - physical_sed_cache.pkl addresses this for subsequent reruns')
print()
print('[Reload guide for this notebook]')
print('  REBUILD_PHYSICAL_TEMPLATES = True   → re-runs slsnni() chain')
print('  REBUILD_PHYSICAL_MAG_GRID  = True   → re-runs synthesize_mag_at_z() grid')
print('  REBUILD_SED_CACHE          = True   → forces slsnni() even if cache exists')
print('  All False                          → loads from disk (fast rerun)')
print()
print('Done.')